[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/tippe_top/tippe_top.ipynb)

# Chiral Tippe Tops

A chiral tippe top can favor one direction of spin. This example releases the same hollow body with equal and opposite spins: one launch climbs onto its tip, while the other stays upright on its shell.

The visible solid and its mass distribution come from the same quadrics. Each quadric maps a point to its polar plane; `surface(point) & point <= 0` selects the material inside. Each mass sample joins its motion under a `Rate` to its position, contributing to the inertia. Friction at the shell and the tip couples spin to rocking.

In homogeneous-coordinate notation, the quadric interior reads as $\mathbf p^{\mathsf T}Q\mathbf p \leq 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial
from itertools import tee
from typing import NamedTuple

import jax
import numpy as np

from numga import stack
from numga.algebras import PGA3D as ga
from numga.backend.jax import JaxContext
from examples.mechanics import lie_integrators as lie
from examples.mechanics.spinning_top import core as contact
from examples.mechanics.tippe_top import render
from examples.util.animation import save_figure, show_animation

In [ ]:
jax.config.update("jax_enable_x64", True)
mv = JaxContext(ga, dtype=np.float64).multivector
Point = ga.gatype.antivector()                                               # Antivector-
Plane = ga.gatype.vector()                                                  # Vector+
Rate = ga.gatype.bivector()                                                 # Bivector+
Forque = ga.gatype.antibivector()                                           # Antibivector-
Motor = ga.gatype.rotor()                                                   # Rotor+
Scalar = ga.gatype.scalar()                                                 # Scalar+
Quadric = ga.gatype((Plane, Point))
Inertia = ga.gatype((Forque, Rate))
InverseInertia = ga.gatype((Rate, Forque))
origin = mv.w.dual()                                                        # [] Point-
up = mv.z.dual()                                                            # [] Point-
axes = stack((mv.x, mv.y, mv.z))                                           # [axes] Plane+
ground = (mv.z * (mv.w & Point) + mv.w * (mv.z & Point)) / 2                # [] Plane+ <- Point-


class Body(NamedTuple):
    shell: Quadric
    cavity: Quadric
    stem: Quadric
    tip: Quadric
    mass: Point
    inertia: Inertia
    inverse_inertia: InverseInertia

    @property
    def contacts(self) -> Quadric:
        """The two surfaces that meet the table: the shell and the tip."""
        return stack((self.shell, self.tip))

## A hollow body

An ellipsoid cavity is offset and turned inside the outer ellipsoid, and breaks through its top, so that the inside shows. Their unequal horizontal axes and their relative orientation make the mass distribution prefer a different rocking direction from the contact geometry. An ellipsoidal stem joins the shell to a spherical tip. All of it has the same density, including the gold-colored stem; overlapping pieces count only once.

In [ ]:
# The outer shell about the origin, the cavity carved from it, open at the top, the stem and the ball at its end.
shell_semiaxes = np.array([1.05, 0.95, 1.01])
cavity_centre = mv.point([-0.05, -0.04, 0.47])
cavity_semiaxes = np.array([0.72, 0.65, 0.59])
cavity_angle = 0.16
cavity_tilt = 0.07
stem_centre = mv.point([0, 0, 0.21])
stem_semiaxes = np.array([0.18, 0.18, 1.02])
tip_centre = mv.point([0, 0, 1.23])
tip_radius = 0.20
sample_spacing = 0.025
sample_lower = np.array([-1.125, -1.125, -1.125])
sample_upper = np.array([1.125, 1.125, 1.525])

# Equal-volume boxes cover the entire solid; each midpoint carries the material in its box.
sample_coordinates = [np.arange(low, high, sample_spacing) + sample_spacing / 2
                      for low, high in zip(sample_lower, sample_upper)]
samples = mv.point(np.stack(np.meshgrid(*sample_coordinates, indexing="ij"), axis=-1).reshape(-1, 3))

In [ ]:
def ellipsoid(centre: Point, frame: Plane, semiaxes: np.ndarray) -> Quadric:
    """The ellipsoid about a point with its axes square to a frame's planes `[axes]`, as the quadric of its surface."""
    through = frame - mv.w * (frame & centre)                                # [axes] Plane+
    return (through * (through & Point) / semiaxes**2).sum(axis=-1) - mv.w * (mv.w & Point)


@jax.jit
def build(shell: Quadric, cavity: Quadric, stem: Quadric, tip: Quadric,
          samples: Point, sample_volume: float) -> Body:
    """The shell carved by the cavity, joined to the stem and the tip: one solid of uniform density, its pieces
    counted once where they overlap."""
    parts = stack((shell, cavity, stem, tip))                                # [parts] Plane+ <- Point-
    in_shell, in_cavity, in_stem, in_tip = (parts[:, None](samples) & samples) <= 0
    material = (in_shell & ~in_cavity) | in_stem | in_tip                    # [samples]
    masses = material * sample_volume
    mass = (samples * masses).sum(axis=-1)                                  # [] Point-
    # Each mass sample joins its velocity to its position, contributing to the body's momentum.
    inertia = samples.regressive(samples.commutator(Rate) * masses).sum(axis=-1)
    return Body(shell, cavity, stem, tip, mass, inertia, inertia.inverse())


cavity_turn = (mv.yz * (-cavity_tilt / 2)).exp() * (mv.xy * (-cavity_angle / 2)).exp()
shell = ellipsoid(origin, axes, shell_semiaxes)
cavity = ellipsoid(cavity_centre, cavity_turn >> axes, cavity_semiaxes)
stem = ellipsoid(stem_centre, axes, stem_semiaxes)
tip = ellipsoid(tip_centre, axes, tip_radius * np.ones(3))
top = build(shell, cavity, stem, tip, samples, sample_spacing**3)
save_figure(render.draw_setup(top), "tippe_top_construction");

## Opposite spins

The shell and the ball tip are two contacting surfaces of one rigid body. The stem stays behind their support envelope, and the cavity leaves the shell's exposed contact points intact. Both launches have the same small lean and spin about the same line through the mass centre.

The negative spin is on the left and the positive spin on the right. The views follow each top across the table, through the stretch where the left one flips, slowed down so that the spin of the chequered shell shows; the chart below follows the whole run. This configuration demonstrates a spin preference for these launches; it does not claim inversion from every release.

In [ ]:
spin_speed = 13.0
spins = spin_speed * np.array([-1., 1.])
lean = 0.1
gravity = 1.0
dynamic_friction = 0.35
static_friction = 1.2 * dynamic_friction
step = 0.004
# Frames close enough that the spin shows, over the run and, slowed down, over the left top's flip: briefly upright,
# then over, and at rest on its tip.
frame_interval = 0.16
n_steps_per_frame = round(frame_interval / step)
run_time = 80.0
n_states = round(run_time / frame_interval) + 1
flip_start, flip_end = 53.0, 65.0
first_frame = round(flip_start / frame_interval)
n_frames = round((flip_end - flip_start) / frame_interval)
frame_duration_ms = 40

In [ ]:
def launch(body: Body, spins: np.ndarray, lean: float) -> tuple[Motor, Rate]:
    """Opposite spins about the mass centre, with the same lean and no initial penetration."""
    tilt = (mv.xz * (lean / 2)).exp()                                        # [] Motor+
    normal = tilt << up                                                      # [] Point-
    supports = contact.lowest_point(body.contacts, normal)                   # [contacts] Point-
    height = (mv.z & (tilt >> supports)).min(axis=0)                         # [] Scalar+
    lift = (-up * origin.inverse() * height / 2).exp()                       # [] Motor+
    motor = (lift * tilt).broadcast_to(spins.shape)
    centre = body.mass.normalized()                                         # [] Point-
    spin_axis = (mv.x - mv.w * (mv.x & centre)) ^ (mv.y - mv.w * (mv.y & centre))
    return motor.retyped(Motor), (spin_axis * spins).cast(Rate).retyped(Rate)


motor, rate = launch(top, spins, lean)

The contact solver corrects each free step along the contacting force lines and limits sliding with Coulomb friction. The corrected motor supplies the next rate, so the same motion drives both contacts. The following cells take about ten seconds, most of it compiling the contact steps and drawing the animation.

In [ ]:
@partial(jax.jit, static_argnames=("n_steps",))
def advance(body: Body, motor: Motor, rate: Rate, gravity: float,
            static: float, dynamic: float, dt: float, n_steps: int) -> tuple[Motor, Rate]:
    """Free rigid motion and XPBD contact corrections advance the top to the next frame."""

    def load(placement: Motor, velocity: Rate) -> Forque:
        return body.mass & (placement << (-up * gravity))

    def substep(index: int, state: tuple[Motor, Rate]) -> tuple[Motor, Rate]:
        before, velocity = state
        predicted, _ = lie.explicit_rk4(before, velocity, body.inertia, body.inverse_inertia, dt, load)
        placement, velocity = contact.project_contacts(before, predicted.normalized(), body.inverse_inertia,
                                                       body.contacts, ground, static, dynamic, dt)
        return placement.retyped(Motor), velocity.retyped(Rate)

    return jax.lax.fori_loop(0, n_steps, substep, (motor, rate))

In [ ]:
def evolve(body: Body, motor: Motor, rate: Rate, gravity: float,
           static: float, dynamic: float, dt: float, n_frames: int,
           stride: int) -> Iterator[tuple[Motor, Rate]]:
    """Yield each launch's placement and rate as the experiment advances."""
    yield motor, rate
    for _ in range(n_frames - 1):
        motor, rate = advance(body, motor, rate, gravity, static, dynamic, dt, stride)
        yield motor, rate


# One stream of states: the flip is drawn as its states come, and the whole run then gives the history.
shown, recorded = tee(evolve(top, motor, rate, gravity, static_friction, dynamic_friction,
                             step, n_states, n_steps_per_frame))
placements = (placement for placement, _ in shown)
show_animation(render.animate(top, placements, first_frame, n_frames), "tippe_top_opposite_spins", frame_duration_ms,
               n_frames)

## Inversion

The orange curve follows the left launch, and the blue curve follows the right. The stem's vertical component is positive while upright, zero when horizontal, and negative after inversion. The falling orange curve records the transfer from shell contact to tip contact; the blue curve stays near its initial orientation.

In [ ]:
vertical = stack([mv.z & (placement >> up) for placement, _ in recorded])   # [states, spins] Scalar+
times = np.arange(n_states) * frame_interval
save_figure(render.history(times, vertical), "tippe_top_inclination");

In [ ]:
# checks
assert vertical[-1, 0] < -0.8
assert vertical[-1, 1] > 0.8